# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [99]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records_copy.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records_copy.csv, course_enrollments_copy.csv, weekly_activity_copy.csv")
    files.upload()
    students = pd.read_csv("student_records_copy.csv")
enroll   = pd.read_csv("course_enrollments_copy.csv")
activity = pd.read_csv("weekly_activity_copy.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [100]:
# TODO: set n_missing_study to the number of blank study_hours_reported values



n_missing_study = students.study_hours_reported.isna().sum()
print(n_missing_study)

255


**Your justification (1–2 sentences):** 

I think I would just add an N/A to each blank column so that we still keep all of the other data. This would be Han's method number 3 and I think it works the best here becasue it keeps all the data but we just dont use the missing data for this column.

In [101]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [102]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups

#before values
print(students['housing'].value_counts(dropna=False))

housing_map = {
    'off-campus': 'Off-Campus',
    'off campus': 'Off-Campus',
    'on-campus': 'On-Campus',
    'on campus': 'On-Campus',
    'with family': 'With Family'
}

students['housing_clean'] = students['housing'].str.strip().str.lower().map(housing_map)

print(students['housing_clean'].value_counts(dropna=False))


housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [103]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [104]:
# TODO

#print(students[(students['age'] > 80) | (students['age'] < 15)])


impossible_ages_unclean = (students['age'] < 16) | (students['age'] > 100)


impossible_ages = students.loc[impossible_ages_unclean, 'age'].tolist()
print("Impossible age values found:", impossible_ages)

n_neg_work = ((students['work_hours_per_week']>80) | (students['work_hours_per_week']< 0)).sum()
print("impossible work hours: ", n_neg_work)


Impossible age values found: [0, 1, 199, -22, 220, 3]
impossible work hours:  4


In [105]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [106]:
# TODO: build students_dedup (one row per student)


print("Exact duplicate rows:", students.duplicated().sum())

# Duplicates based on the student primary key
print("Duplicate student_ids:", students.duplicated(subset=['student_id']).sum())

students_dedup = students.drop_duplicates(subset=['student_id'])

print(len(students_dedup))

Exact duplicate rows: 18
Duplicate student_ids: 27
2000


**Why not de-dupe enroll / activity? (1 sentence):** _..._

In [107]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [108]:
# TODO: build the standardized key and the one-row-per-student "analysis" table

# make the ID an integer
students_dedup['student_id'] = students_dedup['student_id'].astype(str).str.replace("NU-","",regex=False).astype(int)

activity['student_id'] = activity['student_id'].astype(str).str.replace("NU-", "").astype(int)
activity_summary = activity.groupby('student_id')['minutes_active'].sum().reset_index()


analysis = students_dedup.merge(activity_summary, on='student_id', how='left')


In [109]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [110]:
# TODO
enroll['sid'] = enroll['sid'].replace("NU-", "").astype(int)

#counting the matching using the key
set_of_keys = set(students_dedup['student_id'])
matched = enroll['sid'].isin(set_of_keys).sum()

print(matched)


8041


In [111]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [112]:
# TODO


print("Blanks before:", students['enrollment_date'].isna().sum())


students_dedup['enrollment_date'] = pd.to_datetime(
    students_dedup['enrollment_date'],
    format='mixed',
    errors='coerce'
)


analysis['enrollment_date'] = pd.to_datetime(
    analysis['enrollment_date'],
    format='mixed',
    errors='coerce'
)


print("Data type:", analysis['enrollment_date'].dtype)
print("\nSample parsed dates:")
print(analysis['enrollment_date'].dropna().head())


print("\nTotal unparseable / missing dates (NaT):", analysis['enrollment_date'].isna().sum())


Blanks before: 0
Data type: datetime64[us]

Sample parsed dates:
0   2023-09-13
1   2023-10-19
2   2022-11-16
3   2024-05-03
4   2022-10-24
Name: enrollment_date, dtype: datetime64[us]

Total unparseable / missing dates (NaT): 0


In [113]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [114]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]

x = analysis['study_hours_reported']
analysis['study_z'] = (x - x.mean()) / x.std()

analysis['gpa_band'] = pd.cut(analysis['final_gpa'], bins=[-0.01, 1, 2, 3, 4])

print(analysis[['study_z', 'final_gpa', 'gpa_band']].head())

    study_z  final_gpa    gpa_band
0       NaN       1.86  (1.0, 2.0]
1 -0.251817       2.06  (2.0, 3.0]
2  2.241895       3.41  (3.0, 4.0]
3 -0.615483       1.73  (1.0, 2.0]
4 -1.005126       2.37  (2.0, 3.0]


In [115]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [116]:
# TODO: write analysis to northgate_clean.csv

analysis.to_csv('northgate_clean.csv', index=False)



In [117]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing values - A lot of the study hours reported fields were blank. Instead of removing the whole line, I would just put N/A in the field and not count them for study hours but will count for everthing else.
- Housing - There were 3 distict categories but they were all written differently so I just grouped them all into the same categories. EX: on campus and ON-campus
- Impossible values - I removed impossible ages like 3, -22, and 220 as well as negaive hours worked because those are impossible.
- Duplicates - I removed duplicates to make sure the data is not skewed and no single data is counted more than once.
- Key - I took off the "NU-" from the ID to make it an integer so it is easier to work with.
- Dates - I parsed the enrollment dates so they are all in the same format because a lot of them were in completely different formats.


### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [120]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data

import pandas as pd


clean_df = pd.read_csv('northgate_clean.csv')


# 1. Overall Mean GPA
overall_mean_gpa = analysis['final_gpa'].mean()
print(f"Overall Mean GPA: {overall_mean_gpa:.2f}\n")

# 2. Relationship: GPA broken down by housing category
# Use 'housing_clean' (or whatever you named the clean housing column)
housing_col = 'housing_clean' if 'housing_clean' in analysis.columns else 'housing'

housing_gpa = analysis.groupby(housing_col)['final_gpa'].agg(['count', 'mean']).reset_index()
print("GPA by Housing Category:")
print(housing_gpa)

Overall Mean GPA: 2.31

GPA by Housing Category:
  housing_clean  count      mean
0    Off-Campus    932  2.329925
1     On-Campus    584  2.276164
2   With Family    484  2.293285


The clean data makes for a more accurate result and an easier comparison between categorical variables like housing. In this case, there is no statistical significance in where you live to what will be your final GPA. all the categories gover around 2.3 GPA

In [119]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
